# Coordinación observada

Eventos operativos depurados: participantes, herramientas, destinatarios y tiempos. Los textos y registros originales quedan con el ejecutor. Una diferencia de coordinación no identifica su efecto causal.

In [ ]:
from pathlib import Path
import json, os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from native_eval.bundle import COLUMNS, comparison, verify_bundle
bundle = Path(globals().get('BUNDLE', os.environ.get('NATIVE_EVAL_BUNDLE', '.runs/bundle')))
runs = pd.read_csv(bundle / 'runs.csv') if (bundle / 'runs.csv').is_file() else pd.DataFrame(columns=COLUMNS)
print('Sin resultados: no hay corridas de evaluación.' if runs.empty else f'{len(runs)} intentos observados; se muestran también los fallos.')


In [ ]:
events = pd.DataFrame(json.loads((bundle / 'events.json').read_text())) if (bundle / 'events.json').is_file() else pd.DataFrame()
if not events.empty:
    display(events)
    for slot, frame in events.groupby('slot_id'):
        visible = frame.dropna(subset=['actor','time']).copy()
        if visible.empty: continue
        visible['elapsed_seconds'] = visible['time'] - frame['time'].min()
        fig, axis = plt.subplots(figsize=(10, 3))
        for actor, participant in visible.groupby('actor'):
            axis.scatter(participant['elapsed_seconds'], [actor]*len(participant), label=actor, s=15)
        axis.set(title=slot, xlabel='Segundos desde el primer evento', ylabel='Participante')
        plt.tight_layout(); plt.show()
        display(frame.groupby(['actor','operation'], dropna=False).size().rename('events').reset_index())
